# 05 - Neuro-Symbolic Entity Linking & Text2SQL (Batch Processing)

This notebook demonstrates **neuro-symbolic entity linking** and **Text2SQL**
patterns applied to Uruguay's crime-statistics datasets, drawing on the
framework described in [arXiv:2508.13678](https://arxiv.org/abs/2508.13678).

**Model strategy** (A100 GPU):
- **Qwen2.5-7B-Instruct**: Fast Spanish NLP, entity linking, Text2SQL
- **QwQ-32B**: Deep reasoning tasks (NSVIF semantic evaluation, causal analysis)

**Critical design principle**: All LLM inference uses **batch processing** to
maximize GPU utilization. We never process items one-by-one — instead we
compose full batches (all columns at once, all queries at once, all constraints
at once) and send them in a single inference call or minimal batched calls.

### What this notebook covers

1. **Entity Linking** — Match columns across ALL 6 datasets in a single batch prompt
2. **Text2SQL** — Batch multiple Spanish queries into one inference call
3. **Semantic Constraint Evaluation** — Evaluate ALL constraints for a dataset in one pass

In [ ]:
!pip install -q transformers torch accelerate polars duckdb

In [ ]:
import json
from pathlib import Path

import duckdb
import polars as pl
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

In [ ]:
# ---------------------------------------------------------------------------
# Load Models — two-model strategy for A100
# ---------------------------------------------------------------------------
# Qwen2.5-7B for fast entity linking & Text2SQL (high throughput)
# QwQ-32B for deep reasoning / NSVIF semantic evaluation
#
# We load Qwen first (smaller), use it for batch tasks, then swap to QwQ
# for the reasoning-heavy task. This avoids VRAM fragmentation.

FAST_MODEL = "Qwen/Qwen2.5-7B-Instruct"     # entity linking, Text2SQL
REASONING_MODEL = "Qwen/QwQ-32B"              # NSVIF semantic evaluation


def load_model(model_id: str):
    """Load a model with optimal A100 settings."""
    print(f"Loading {model_id}...")
    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.bfloat16,   # A100 native dtype for max throughput
        device_map="auto",
        trust_remote_code=True,
    )
    print(f"  Loaded on {next(model.parameters()).device}")
    return tokenizer, model


def unload_model(model, tokenizer):
    """Free GPU memory before loading next model."""
    del model
    del tokenizer
    torch.cuda.empty_cache()
    print("  GPU memory freed.")


def batch_inference(tokenizer, model, prompts: list[str], max_new_tokens: int = 1024) -> list[str]:
    """Run batched inference over multiple prompts in a SINGLE forward pass.

    This is the critical optimization: instead of calling the model N times
    (one per prompt), we batch all prompts together and process them in one
    GPU pass. This maximizes GPU utilization and avoids the overhead of
    repeated model calls.
    """
    # Apply chat template to each prompt
    texts = []
    for prompt in prompts:
        messages = [{"role": "user", "content": prompt}]
        text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        texts.append(text)

    # Tokenize as a batch with padding
    inputs = tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
    ).to(model.device)

    # Single batched forward pass
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
        )

    # Decode each output, skipping input tokens
    results = []
    for i, ids in enumerate(output_ids):
        input_len = inputs["input_ids"][i].ne(tokenizer.pad_token_id or 0).sum()
        generated = ids[input_len:]
        text = tokenizer.decode(generated, skip_special_tokens=True).strip()
        results.append(text)

    return results


def single_inference(tokenizer, model, prompt: str, max_new_tokens: int = 2048) -> str:
    """Single prompt inference — used only when the prompt itself is a mega-batch
    (i.e., all work is packed into one prompt to avoid multiple calls)."""
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )
    generated = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()


# Load fast model first
tokenizer, model = load_model(FAST_MODEL)

In [ ]:
# ---------------------------------------------------------------------------
# Load ALL datasets at once (batch loading)
# ---------------------------------------------------------------------------
DATA_DIR = Path("../data/processed/tabular")

DATASETS = {
    "delitos_denunciados": "delitos_denunciados.parquet",
    "violencia_domestica": "violencia_domestica.parquet",
    "delitos_sexuales": "delitos_sexuales.parquet",
    "homicidios_mujeres": "homicidios_mujeres.parquet",
    "medidas_alternativas": "medidas_alternativas.parquet",
    "sistema_carcelario": "sistema_carcelario.parquet",
}

# Synthetic fallback schemas (used if parquet files don't exist yet)
SYNTHETIC_SCHEMAS = {
    "delitos_denunciados": ["anio", "mes", "trimestre", "departamento", "seccional", "titulo", "subtitulo", "fecha", "sexo", "cantidad"],
    "violencia_domestica": ["ano", "departamento_hecho", "titulo", "jurisdiccion", "sexo_victima", "sexo_agresor", "fecha_denuncia", "tipo_violencia", "total"],
    "delitos_sexuales": ["anio", "departamento", "jurisdiccion", "titulo", "sexo_victima", "fecha_denuncia", "total"],
    "homicidios_mujeres": ["anio", "departamento", "vinculo", "fecha", "tipo_homicidio", "total"],
    "medidas_alternativas": ["anio", "mes", "departamento", "tipo_medida", "genero", "medio_control", "cantidad"],
    "sistema_carcelario": ["anio", "mes", "establecimiento", "sexo", "grupo_edad", "delito", "nacionalidad", "cantidad"],
}

all_columns = {}
dataframes = {}

for name, filename in DATASETS.items():
    path = DATA_DIR / filename
    if path.exists():
        df = pl.read_parquet(path)
        all_columns[name] = df.columns
        dataframes[name] = df
        print(f"  {name}: {df.shape} loaded")
    else:
        all_columns[name] = SYNTHETIC_SCHEMAS[name]
        print(f"  {name}: using synthetic schema ({len(SYNTHETIC_SCHEMAS[name])} cols)")

print(f"\nTotal datasets: {len(all_columns)}")

In [ ]:
# ---------------------------------------------------------------------------
# BATCH Entity Linking — ALL 6 datasets in ONE inference call
# ---------------------------------------------------------------------------
# Instead of comparing datasets pairwise (15 pairs = 15 LLM calls),
# we send ALL column schemas in a SINGLE prompt and ask the model to
# identify ALL cross-dataset matches at once.
#
# This is the batch-processing lesson from project-1: never item-by-item.

schema_block = "\n".join(
    f"Dataset '{name}': {json.dumps(cols)}"
    for name, cols in all_columns.items()
)

entity_linking_prompt = (
    "You are a data integration expert working with Uruguayan crime statistics.\n"
    "\n"
    "Here are the column schemas for ALL 6 datasets from Uruguay's Interior Ministry:\n\n"
    + schema_block +
    "\n\n"
    "Task: Identify ALL semantically equivalent columns ACROSS datasets in one pass.\n"
    "Consider:\n"
    "- Different spellings of the same concept (anio vs ano vs year)\n"
    "- Prefixed/suffixed variants (departamento vs departamento_hecho)\n"
    "- Same concept different names (sexo vs genero vs sexo_victima)\n"
    "- Temporal columns (fecha, fecha_denuncia, anio+mes)\n"
    "- Count columns (cantidad vs total)\n"
    "\n"
    "Return a JSON list of match groups. Each group has:\n"
    '  "concept": canonical name for this dimension\n'
    '  "matches": list of {"dataset": str, "column": str}\n'
    '  "join_type": "exact" | "fuzzy" | "derived"\n'
    '  "notes": any caveats\n'
    "\n"
    "Include ALL meaningful matches. This is for building a unified data model."
)

print(f"Sending ALL 6 dataset schemas in ONE prompt ({len(entity_linking_prompt)} chars)...")
linking_result = single_inference(tokenizer, model, entity_linking_prompt, max_new_tokens=2048)
print("\n" + linking_result)

In [ ]:
# ---------------------------------------------------------------------------
# BATCH Text2SQL — ALL queries in ONE inference call
# ---------------------------------------------------------------------------
# Instead of sending questions one-by-one (N calls), we pack ALL questions
# into a single prompt and get ALL SQL queries back in one GPU pass.

# Pick a dataset to query
if "delitos_denunciados" in dataframes:
    df_query = dataframes["delitos_denunciados"]
else:
    df_query = pl.DataFrame({
        "departamento": ["Montevideo"] * 5 + ["Canelones"] * 5,
        "anio": [2022, 2022, 2023, 2023, 2023, 2022, 2022, 2023, 2023, 2023],
        "titulo": ["Hurto", "Rapina", "Hurto", "Rapina", "Lesiones",
                   "Hurto", "Rapina", "Hurto", "Rapina", "Lesiones"],
        "cantidad": [1200, 350, 1300, 380, 200, 800, 150, 850, 170, 120],
    })

con = duckdb.connect()
con.register("delitos", df_query.to_pandas())
schema_info = con.execute("DESCRIBE delitos").fetchdf().to_string()

# All questions in one batch
questions = [
    "¿Cuántos delitos se denunciaron en Montevideo en 2023?",
    "¿Cuál es el departamento con más delitos en total?",
    "¿Cuántos registros hay por tipo de delito?",
    "¿Cuál fue la evolución de rapiñas entre 2022 y 2023?",
    "¿Qué porcentaje del total representan los hurtos?",
]

numbered_questions = "\n".join(f"{i+1}. {q}" for i, q in enumerate(questions))

batch_sql_prompt = (
    "You are a SQL expert. Given this DuckDB table schema:\n\n"
    f"Table: delitos\n{schema_info}\n\n"
    f"Generate SQL queries for ALL of the following questions (in Spanish).\n"
    f"Return ONLY the SQL for each, numbered to match:\n\n"
    f"{numbered_questions}\n\n"
    "Format your response as:\n"
    "1. <SQL>\n2. <SQL>\n...\n"
    "No explanations, just numbered SQL queries."
)

print(f"Sending {len(questions)} questions in ONE prompt...")
batch_sql_result = single_inference(tokenizer, model, batch_sql_prompt, max_new_tokens=1024)
print("\nGenerated SQL batch:")
print(batch_sql_result)

# Parse and execute each SQL
print("\n" + "=" * 60)
print("EXECUTING GENERATED SQL")
print("=" * 60)

import re
sql_blocks = re.findall(r'\d+\.\s*(?:```sql\s*)?(.+?)(?:```|\n\d+\.|$)', batch_sql_result, re.DOTALL)

for i, (question, sql) in enumerate(zip(questions, sql_blocks)):
    sql_clean = sql.strip().strip('`').strip()
    if sql_clean.lower().startswith('sql'):
        sql_clean = sql_clean[3:].strip()
    print(f"\nQ{i+1}: {question}")
    print(f"SQL: {sql_clean}")
    try:
        result = con.execute(sql_clean).fetchdf()
        print(result.to_string(index=False))
    except Exception as e:
        print(f"  Execution error: {e}")

con.close()

In [ ]:
# ---------------------------------------------------------------------------
# Swap to reasoning model for NSVIF semantic evaluation
# ---------------------------------------------------------------------------
# Free Qwen2.5-7B, load QwQ-32B for the reasoning-heavy task.
# QwQ-32B fits on A100 in bfloat16 (~64GB, tight on 40GB A100,
# use 4-bit quantization as fallback).

unload_model(model, tokenizer)

try:
    tokenizer_r, model_r = load_model(REASONING_MODEL)
except Exception as e:
    print(f"QwQ-32B failed ({e}), falling back to Qwen2.5-7B for reasoning too.")
    tokenizer_r, model_r = load_model(FAST_MODEL)

In [ ]:
# ---------------------------------------------------------------------------
# BATCH Semantic Constraint Evaluation (NSVIF Semantic Agent)
# ---------------------------------------------------------------------------
# Instead of evaluating constraints one-by-one, we send ALL values and ALL
# constraints in a SINGLE prompt. The reasoning model evaluates the entire
# batch and returns structured results.
#
# This mirrors the project-1 lesson: process dataset-by-dataset, not
# record-by-record.

OFFICIAL_TAXONOMY = [
    "Hurto", "Rapiña", "Homicidio", "Lesiones",
    "Violencia doméstica", "Daño", "Estafa",
    "Abigeato", "Copamiento", "Secuestro",
    "Abuso sexual", "Violación", "Atentado violento al pudor",
]

# Batch ALL test values together — not one at a time
test_values = [
    "Hurto",
    "Robo con violencia",
    "Homicidio",
    "Asalto a mano armada",
    "Violencia de género",
    "Abuso sexual con contacto corporal",
    "Femicidio",
    "Tentativa de rapiña",
    "Lesiones graves",
    "Maltrato animal",
    "Hurto de ganado",
    "Acoso sexual",
]

semantic_prompt = (
    "You are a legal expert on Uruguayan criminal law. Think step by step.\n\n"
    "The official crime taxonomy used by Uruguay's Ministry of the Interior is:\n"
    + json.dumps(OFFICIAL_TAXONOMY) +
    "\n\n"
    "Evaluate ALL of the following crime descriptions IN ONE PASS.\n"
    "For EACH value determine:\n"
    "1. exact_match: does it exactly match a taxonomy entry? (true/false)\n"
    "2. closest_match: closest official category\n"
    "3. confidence: 0-1\n"
    "4. reasoning: why this mapping (consider Uruguayan legal context)\n\n"
    "Crime descriptions to evaluate (ALL AT ONCE):\n"
    + json.dumps(test_values, ensure_ascii=False) +
    "\n\n"
    "Return a JSON list with one object per input value.\n"
    "Keys: input, exact_match, closest_match, confidence, reasoning"
)

print(f"Evaluating {len(test_values)} values in ONE reasoning pass...\n")
semantic_result = single_inference(tokenizer_r, model_r, semantic_prompt, max_new_tokens=3000)
print(semantic_result)

In [ ]:
# Cleanup
unload_model(model_r, tokenizer_r)
print("All models unloaded. GPU memory freed.")

## Summary: Batch Neuro-Symbolic Patterns

This notebook demonstrated three **LLM -> Symbolic** patterns from
[arXiv:2508.13678](https://arxiv.org/abs/2508.13678), all using **batch processing**
to maximize A100 GPU utilization:

| Task | Model | Batch Strategy | Items/Call |
|---|---|---|---|
| **Entity Linking** | Qwen2.5-7B | All 6 dataset schemas in ONE prompt | 6 datasets |
| **Text2SQL** | Qwen2.5-7B | All 5 questions in ONE prompt | 5 queries |
| **Semantic Constraints** | QwQ-32B | All 12 values in ONE prompt | 12 values |

### Batch Processing Principle

From project-1 we learned: **never process items one-by-one through the LLM**.
The GPU is most efficient when saturated with work. Three strategies:

1. **Mega-prompt**: Pack all work into a single prompt (entity linking, semantic eval)
2. **Batched forward pass**: Use `model.generate()` with padded batch inputs (Text2SQL)
3. **Model swapping**: Load fast model -> batch fast tasks -> unload -> load reasoning model -> batch reasoning tasks

### Model Selection Rationale

| Model | Task Type | Why |
|---|---|---|
| **Qwen2.5-7B-Instruct** | Spanish NLP, SQL generation | Best Spanish support, fast throughput, fits easily on A100 |
| **QwQ-32B** | Deep reasoning, legal analysis | Strong chain-of-thought, inherits Qwen's Spanish, fits A100 in bf16 |

Kimi-K2 was considered for reasoning but its Spanish support is unproven (CN/EN focus).
DeepSeek-R1 is a viable alternative if QwQ-32B doesn't fit your A100 variant.